# Performing Domain Analysis with AI  
## Leveraga SA Model, Write Up, and Results on Strategy to develop a first cut Architecture

First we load up the PEAK Model

### Model-Specific Code (Do Not Modify)

This section contains code that is specific to the system model. It is updated only when the model is changed and should not require user modifications under normal circumstances.

If a new model is introduced, ensure this section is reviewed and updated as needed.


In [1]:
#!pip install --upgrade git+https://github.com/tkSDISW/Capella_Tools 
import capellambse.decl

from capella_tools import capellambse_helper

from IPython import display as diag_display
resources = {
    "PEAK": "PEAK/PEAK",
}
path_to_model = "../PEAK.aird"
model = capellambse.MelodyModel(path_to_model, resources=resources)
from capella_tools import Pub4C
# Instantiate the class with the traceability file
traceability_store = Pub4C.Traceability_Store("../PEAK/traceability")


## 🔄 Embedding Generation Process

Next we load evaluate whether we ne to update the embedings. If there older than model we will update thems. 🚀


In [2]:

from capella_tools  import capella_embeddings_manager

# Generate embeddings for all objects
model_embedding_manager = capella_embeddings_manager.EmbeddingManager()

embedding_file = "embeddings.json" 
model_embedding_manager.set_files( path_to_model , embedding_file)

model_embedding_manager.create_model_embeddings(model)

OpenAI API Key retrieved successfully.
Loading embeddings
embeddings loaded.


## 🎯 Define the model element or diagram for analysis.

We use the embedding to locate a diagram that will be used for basis of analysis. It a OCB diagrams with relations to all the functional chains. 

> 💡 **Tip:** If you're unsure about the model structure, review the documentation or refer to the model's diagrams for additional guidance.


In [3]:

selected_objects = model_embedding_manager.query_and_select_top_objects("[SAB] PEAK", top_n=1)



This is a list of ranked Objects Based on Query:
Index: 0, Name: PEAK, Similarity: 0.60, Type: SystemComponent, Phase: System Analysis SA, Source: , Target: 


## 📝 Generate Structured Input File 

We then generate the structured input file for all the matching objects and there related objects.
A file "capella_model.yaml" is written it you want to look at it. 

In [4]:
#Workflow
from capella_tools import capellambse_yaml_manager
yaml_handler = capellambse_yaml_manager.CapellaYAMLHandler()
   
#Generate YAML for the logical component and append to the file
for object in  selected_objects :  
    yaml_handler.generate_yaml(model.by_uuid(object["uuid"]))  

yaml_handler.generate_traceability_related_objects(model,traceability_store)

#yaml_handler.display()
yaml_handler.generate_yaml_referenced_objects()
#yaml_handler.display()

yaml_handler.write_output_file()


In [6]:
import os
from openai import OpenAI
from IPython.core.display import HTML
from IPython.display import display, clear_output, Markdown, IFrame
from capella_tools  import Open_AI_RAG_manager


#print(object)
# Step 1: Get YAML content
yaml_content = yaml_handler.get_yaml_content()

# Step 2: Invoke ChatGPT for analysis
analyzer = Open_AI_RAG_manager.ChatGPTAnalyzer(yaml_content)


OpenAI API Key retrieved successfully.


## ⚙️ Execute  Prompt
Now we load up the PEAK System document. 

Then execute the prompt that will use the model and system document. 

In [7]:
analyzer.add_text_file_to_messages("PEAK System.docx")
prompt = """
Review model and document and see it there is any requirement to handle the removal of salt from water and state source.
"""
analyzer.follow_up_prompt(prompt)
chatgpt_response = analyzer.get_response()

✅ File `PEAK System.docx` added to messages for analysis.


**Your prompt:** 
Review model and document and see it there is any requirement to handle the removal of salt from water and state source.


**ChatGPT Response:**

The requirement for handling the removal of salt from water is specified both in the YAML file and the PEAK System document. Within the PEAK System document, the section on water purification explicitly states that the system must have the capability to produce potable water from fresh, brackish, or saltwater. This involves a requirement for salt removal. Here is the relevant excerpt:

&gt; The water purification equipment must be capable of producing potable water from fresh, brackish, or salt water and must abide by the following parameters:
&gt; - Includes filtration system, distribution capability, and storage container

In the YAML system model, while specific methods for salt removal are not mentioned, the responsibility for water treatment comes under the "Provide Water" function. Here is how this is outlined:

```yaml
- name: Provide Water
  type: SystemFunction
  primary_uuid: 8b849bf6-ba52-49f4-a0bf-858f96b2e501
  description: ""
  allocated from :
  - name: PEAK
    ref_uuid: ddc7faf6-4ef7-4adb-9c7b-8a55f0958947
  functions owned:
  inputs:
   - name: inManage Water Production
     ref_uuid: e3ecba4d-5d3b-4245-90f8-afffc1c288f1
     exchanges:
      - name: Manage Water Production
        ref_uuid:  40e5ef84-f287-401e-a0c9-b9325845553b
        source_function_name: Operate System
        ref_uuid: a1415442-7d05-4d2d-aac5-7b5c4f70b41f
        target_function_name: Provide Water
        ref_uuid: 8b849bf6-ba52-49f4-a0bf-858f96b2e501
  outputs:
   - name: outFiltered Water
     ref_uuid: f9b1291c-4d02-4a03-b824-9c927765f103
     exchanges:
      - name: Filtered Water
        ref_uuid:  ef016c7f-5811-4f4e-b5b2-4c6d6cfe07d1
        source_function_name: Provide Water
        ref_uuid: 8b849bf6-ba52-49f4-a0bf-858f96b2e501
        target_function_name: Receive Water
        ref_uuid: 65c083b0-4004-4497-9870-cd2109b4b6c5
```

However, the exact method or technology for salt removal is implied rather than explicitly detailed in the YAML file, with an emphasis on filtering and filtration processes:

```yaml
applied property value groups:
- name: Water Production Rate
  ref_uuid: d6539966-0393-4b52-939a-b1481e6d9e8c
```

In summary, the requirement is directly stated in the document to handle salt removal from water, stemming from the context of the desired capabilities of the PEAK system, with the source being the operational-user community's identification of essential tasks to resolve the operational problem.

**Token Usage Info:**

Tokens used: prompt=17522, completion=691, total=18213

## 💬 Launch Interactive Chat on Structured Input




In [8]:
analyzer.add_text_file_to_messages("PEAK System.docx")
prompt = """
What would be the technological impact of having to remove salt from water. How much bigger a problem of cost versus equipment. 
"""
analyzer.follow_up_prompt(prompt)
chatgpt_response = analyzer.get_response()


ERROR! Session/line number was not unique in database. History logging moved to new session 502
✅ File `PEAK System.docx` added to messages for analysis.


**Your prompt:** 
What would be the technological impact of having to remove salt from water. How much bigger a problem of cost versus equipment. 


**ChatGPT Response:**

Removing salt from water to produce potable water is commonly referred to as desalination. The technological impact and cost associated with implementing desalination capabilities, especially in a system like the PEAK JCTD that relies on commercial off-the-shelf technology, can be significant. Here's a breakdown of the factors related to cost versus equipment:

### Technological Impact:

1. **Complexity**: 
   - Desalination involves complex processes such as reverse osmosis or distillation, which require specialized equipment.
   - Incorporating such technology could complicate the system design, increasing the need for skilled operators and regular maintenance.

2. **Energy Consumption**:
   - Desalination is energy-intensive. The integration of this process into a portable kit demands a reliable and efficient power supply, primarily from renewable sources with a fossil fuel backup.

3. **Size and Weight**:
   - Desalination equipment, specifically membrane-based or evaporative components, can be bulky and heavy, challenging the constraint of having a compact, man-portable system.

4. **Waste Management**:
   - Desalination produces brine as a byproduct, necessitating additional processes or equipment for disposal, which can further increase system complexity.

### Cost Considerations:

1. **Capital Costs**:
   - Initial costs for purchasing desalination equipment, which are generally higher than other water filtration systems due to the complexity and materials used (e.g., high-pressure pumps, durable membranes).

2. **Operational Costs**:
   - Includes costs for energy to run desalination processes, replacement parts (e.g., membranes), and regular maintenance to ensure system efficiency.
   - Operator training costs might increase as the complexity of the system demands higher skill levels.

3. **Maintenance Costs**:
   - Regular maintenance of membranes and other components is critical to avoid fouling and to maintain performance, impacting ongoing costs.

### Equipment Versus Cost Trade-Off:

- **Higher Initial Investment**: The necessity for desalination would increase upfront costs due to specialized equipment. This investment might be justifiable if the operational theater is predominantly coastal or involves saline water sources.

- **Operating Cost Balance**: While renewable energy could power the desalination process, ensuring energy sufficiency without increasing the fossil fuel dependency of the system can be challenging and costly.

- **Cost of Simplicity**: Alternative solutions, such as using stand-alone or deployable desalination units that can be separately deployed, may reduce integration complexity at the expense of increased logistical and operational burdens.

### Conclusion:

Incorporating desalination into the PEAK system will significantly impact its technological and financial aspects. Careful trade-offs between capability and constraints are required, with a strong emphasis needed on renewable power efficiency, system miniaturization, and maintenance simplification to mitigate overall costs while maintaining operational effectiveness. Considering the stakeholder expectations of leveraging existing technology and limiting development, the focus should remain on cost-effective and reliable desalination solutions.

**Token Usage Info:**

Tokens used: prompt=19235, completion=590, total=19825

In [ ]:
print("Done")

# 